In [1]:
!pip install transformers torch pandas numpy tqdm scikit-learn lightgbm flask --quiet
!pip install tabulate
print("Done")

Done


In [2]:
CONFIG = {
    "MODEL_NAME":         "vinai/phobert-base-v2",
    "MAX_LENGTH":         128,
    "BATCH_SIZE":         16,
    "EPOCHS":             3,
    "LEARNING_RATE":      2e-5,
    "VECTOR_DIM":         768,
    "OUTPUT_DIR":         "./phobert_foodtour",
    "TRENDING_WEIGHT":    0.7,
    "TRENDING_THRESHOLD": 1.5,
    "ES_WEIGHT_MAX":      10000,
    "ES_WEIGHT_MIN":      1,
    "EPSILON":            1e-6,
}
print("Config loaded")

Config loaded


In [3]:
from google.colab import files

print("Upload products.csv")
up = files.upload()
for fn, data in up.items():
    with open(fn, 'wb') as f: f.write(data)

print("\nUpload search_analytics.csv")
up2 = files.upload()
for fn, data in up2.items():
    with open(fn, 'wb') as f: f.write(data)

Upload products.csv


Saving products.csv to products.csv

Upload search_analytics.csv


Saving search_analytics.csv to search_analytics.csv


In [4]:
import pandas as pd
import numpy as np
import json
import re
import random
from collections import Counter, defaultdict
from datetime import datetime, timedelta

_VN_MAP = {
    'à':'a','á':'a','ả':'a','ã':'a','ạ':'a',
    'â':'a','ầ':'a','ấ':'a','ẩ':'a','ậ':'a',
    'ă':'a','ằ':'a','ắ':'a','ẳ':'a','ặ':'a',
    'è':'e','é':'e','ẻ':'e','ẽ':'e','ẹ':'e',
    'ê':'e','ề':'e','ế':'e','ể':'e','ệ':'e',
    'ì':'i','í':'i','ỉ':'i','ị':'i',
    'ò':'o','ó':'o','ỏ':'o','õ':'o','ọ':'o',
    'ô':'o','ồ':'o','ố':'o','ổ':'o','ộ':'o',
    'ơ':'o','ờ':'o','ớ':'o','ở':'o','ợ':'o',
    'ù':'u','ú':'u','ủ':'u','ụ':'u',
    'ư':'u','ừ':'u','ứ':'u','ử':'u','ự':'u',
    'ỳ':'y','ý':'y','ỷ':'y','ỹ':'y','ỵ':'y','đ':'d',
}

def remove_vn_accent(text):
    if not text: return ''
    return ''.join(_VN_MAP.get(c, c) for c in text.lower())

def parse_pipe(x):
    """Parse pipe-separated string thành list. VD: 'hot|traditional' → ['hot','traditional']"""
    s = str(x).strip() if x and str(x).strip() not in ('', 'nan') else ''
    if not s:
        return []
    return [t.strip() for t in s.split('|') if t.strip()]

def build_product_text(row):
    name = str(row.get('name') or '')
    desc = str(row.get('description') or '')
    tags = row.get('tags_str', '')
    ingredients = row.get('ingredients_str', '')
    category = str(row.get('category_name') or '')
    return f"{name} {desc} {tags} {ingredients} {category}".strip()

print("Helpers ready")

Helpers ready


In [5]:
from tabulate import tabulate

# engine='python' xử lý được các edge case mà C parser bỏ sót
# (vd: field chứa newline, quote lồng nhau, dấu phẩy trong text)
df_products = pd.read_csv(
    "products.csv",
    encoding="utf-8-sig",
    engine="python",
    on_bad_lines="warn",   # bỏ qua row lỗi thay vì crash, in warning
)
df_products.columns = [c.strip().lower().replace(' ', '_') for c in df_products.columns]

# Fillna cho các cột text
for col in ['description', 'tags', 'ingredients', 'nutrition_info', 'category_name', 'image_urls']:
    if col in df_products.columns:
        df_products[col] = df_products[col].fillna('')

# Parse pipe-separated arrays → list (vd: "hot|traditional" → ["hot","traditional"])
df_products['tags_clean']        = df_products['tags'].apply(parse_pipe)
df_products['ingredients_clean'] = df_products['ingredients'].apply(parse_pipe)
df_products['images_clean']      = df_products['image_urls'].apply(parse_pipe)

df_products['tags_str']        = df_products['tags_clean'].apply(lambda x: ' '.join(x))
df_products['ingredients_str'] = df_products['ingredients_clean'].apply(lambda x: ' '.join(x))
df_products['product_text']    = df_products.apply(build_product_text, axis=1)

print(f"DONE: {len(df_products)} products")
print(tabulate(
    df_products[['id','name','price','rating','shop_name','category_name','tags_str','ingredients_str']].head(36),
    headers='keys', tablefmt='fancy_grid', showindex=False
))

DONE: 474 products
╒══════╤═══════════════════════════════════╤═════════╤══════════╤═════════════════════════════╤═════════════════╤═══════════════════════════════════════╤════════════════════════════════════════════════════╕
│   id │ name                              │   price │   rating │ shop_name                   │ category_name   │ tags_str                              │ ingredients_str                                    │
╞══════╪═══════════════════════════════════╪═════════╪══════════╪═════════════════════════════╪═════════════════╪═══════════════════════════════════════╪════════════════════════════════════════════════════╡
│  141 │ Bánh mì xíu mại                   │   21000 │      3.7 │ Quán Bánh cuốn Gia Truyền B │ Bánh mì         │ banh_mi xiu_mai sot                   │ bánh mì xíu mại sốt cà hành phi                    │
├──────┼───────────────────────────────────┼─────────┼──────────┼─────────────────────────────┼─────────────────┼───────────────────────────────────────┼

In [6]:
df_analytics = pd.read_csv("search_analytics.csv", encoding='utf-8-sig')
df_analytics.columns = [c.strip().lower().replace(' ', '_') for c in df_analytics.columns]
df_analytics['query_text']  = df_analytics.get('query_text', pd.Series(dtype=str)).fillna('')
df_analytics['searched_at'] = pd.to_datetime(df_analytics.get('searched_at'), errors='coerce')
df_analytics = df_analytics[df_analytics['query_text'].str.strip() != '']

print(f"{len(df_analytics)} analytics rows")
print(f"Clicks: {df_analytics['clicked_product_id'].notna().sum()}")

1024 analytics rows
Clicks: 546


In [7]:
import torch
from transformers import AutoTokenizer, AutoModel
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])
model     = AutoModel.from_pretrained(CONFIG["MODEL_NAME"]).to(device)
print(f"PhoBERT loaded")

def mean_pooling(model_output, attention_mask):
    token_emb = model_output.last_hidden_state
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
    return torch.sum(token_emb * mask_expanded, 1) / torch.clamp(mask_expanded.sum(1), min=1e-9)

def get_embedding(text, model, tokenizer, device):
    enc = tokenizer(
        text, max_length=CONFIG['MAX_LENGTH'],
        padding='max_length', truncation=True, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        out = model(**enc)
    emb = mean_pooling(out, enc['attention_mask'])
    emb = nn.functional.normalize(emb, p=2, dim=1)
    return emb.cpu().numpy()[0].tolist()

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

PhoBERT loaded


In [8]:
from torch.utils.data import Dataset, DataLoader

df_clicks = df_analytics[df_analytics['clicked_product_id'].notna()].copy()
df_clicks['clicked_product_id'] = df_clicks['clicked_product_id'].astype(int)

product_text_map = dict(zip(df_products['id'].astype(int), df_products['product_text']))
product_id_list = list(product_text_map.keys())

pairs = []
for _, row in df_clicks.iterrows():
    query = str(row['query_text']).strip()
    pid   = int(row['clicked_product_id'])
    if pid in product_text_map and query:
        pairs.append({'query': query, 'product_text': product_text_map[pid], 'label': 1.0})

neg_pairs = [{'query': p['query'], 'product_text': product_text_map[random.choice(product_id_list)], 'label': 0.0} for p in pairs]
all_pairs = pairs + neg_pairs
random.shuffle(all_pairs)
print(f"{len(all_pairs)} pairs ({len(pairs)} positive, {len(neg_pairs)} negative)")

class SearchDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len):
        self.pairs = pairs; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        p = self.pairs[idx]
        q_enc = self.tokenizer(p['query'], max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        d_enc = self.tokenizer(p['product_text'], max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'q_input_ids': q_enc['input_ids'].squeeze(), 'q_attention_mask': q_enc['attention_mask'].squeeze(),
            'd_input_ids': d_enc['input_ids'].squeeze(), 'd_attention_mask': d_enc['attention_mask'].squeeze(),
            'label': torch.tensor(p['label'], dtype=torch.float),
        }

dataloader = DataLoader(SearchDataset(all_pairs, tokenizer, CONFIG['MAX_LENGTH']), batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
print(f"DataLoader ready: {len(dataloader)} batches")

1042 pairs (521 positive, 521 negative)
DataLoader ready: 66 batches


In [9]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=CONFIG['LEARNING_RATE'])
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0,
    num_training_steps=len(dataloader) * CONFIG['EPOCHS'])
loss_fn = nn.CosineEmbeddingLoss()

model.train()
for epoch in range(CONFIG['EPOCHS']):
    total_loss = 0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}"):
        q_out = model(input_ids=batch['q_input_ids'].to(device), attention_mask=batch['q_attention_mask'].to(device))
        d_out = model(input_ids=batch['d_input_ids'].to(device), attention_mask=batch['d_attention_mask'].to(device))
        q_emb = nn.functional.normalize(mean_pooling(q_out, batch['q_attention_mask'].to(device)), p=2, dim=1)
        d_emb = nn.functional.normalize(mean_pooling(d_out, batch['d_attention_mask'].to(device)), p=2, dim=1)
        labels = batch['label'].to(device)
        target = torch.where(labels > 0.5, torch.ones_like(labels), torch.full_like(labels, -1.0))
        loss = loss_fn(q_emb, d_emb, target)
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} - Avg Loss: {total_loss/len(dataloader):.4f}")

print("Training complete!")
# Loss mong đợi: Epoch 1 ~0.3, Epoch 3 ~0.1


Epoch 1/3: 100%|██████████| 66/66 [00:42<00:00,  1.57it/s]


Epoch 1 - Avg Loss: 0.4845


Epoch 2/3: 100%|██████████| 66/66 [00:41<00:00,  1.57it/s]


Epoch 2 - Avg Loss: 0.4028


Epoch 3/3: 100%|██████████| 66/66 [00:46<00:00,  1.41it/s]

Epoch 3 - Avg Loss: 0.3342
Training complete!


In [10]:
import os, shutil
from google.colab import files

os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)
model.save_pretrained(CONFIG['OUTPUT_DIR'])
tokenizer.save_pretrained(CONFIG['OUTPUT_DIR'])
shutil.make_archive("phobert_foodtour", 'zip', CONFIG['OUTPUT_DIR'])
files.download("phobert_foodtour.zip")
print("Downloaded phobert_foodtour.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded phobert_foodtour.zip


In [11]:
model.eval()
print("Generating embeddings...")

all_embeddings = []
for _, row in tqdm(df_products.iterrows(), total=len(df_products)):
    all_embeddings.append(get_embedding(row['product_text'], model, tokenizer, device))

df_products['embedding'] = all_embeddings
print(f"{len(all_embeddings)} embeddings generated, dim={len(all_embeddings[0])}")

Generating embeddings...


100%|██████████| 474/474 [00:05<00:00, 83.02it/s]

474 embeddings generated, dim=768


In [12]:
def normalize_query(text):
    if not text: return ''
    text = text.lower().strip()
    text = remove_vn_accent(text)
    text = re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', '', text))
    return text.strip()

now = df_analytics['searched_at'].max().replace(tzinfo=None)
cut_1d  = now - timedelta(days=1)
cut_7d  = now - timedelta(days=7)
cut_15d = now - timedelta(days=15)

stats = defaultdict(lambda: {'texts': [], 'total': 0, 'c1d': 0, 'c7d': 0, 'c15d': 0, 'last': None})
for _, row in df_analytics.iterrows():
    raw_q = str(row.get('query_text', '')).strip()
    if not raw_q: continue
    norm = normalize_query(raw_q)
    ts = row.get('searched_at')
    if pd.isna(ts): ts = now
    s = stats[norm]
    s['texts'].append(raw_q); s['total'] += 1
    if ts >= cut_1d:  s['c1d'] += 1
    if ts >= cut_7d:  s['c7d'] += 1
    if ts >= cut_15d: s['c15d'] += 1
    if s['last'] is None or ts > s['last']: s['last'] = ts

suggestion_docs = []
epsilon = CONFIG['EPSILON']
print("Generating suggestion embeddings...")
for norm, s in tqdm(stats.items()):
    recent_count = s['c15d']
    avg_recent = recent_count / 15.0 if recent_count > 0 else 0
    multiplier = (s['c1d'] + s['c7d']*0.5) / (avg_recent + epsilon) if avg_recent >= epsilon else 1.0
    final_score = max(recent_count * (1 + CONFIG['TRENDING_WEIGHT'] * (multiplier - 1)), 0.0)
    best_text = Counter(s['texts']).most_common(1)[0][0]
    emb = get_embedding(best_text, model, tokenizer, device)
    suggestion_docs.append({
        'query_text': best_text, 'query_normalized': norm,
        'no_accent': remove_vn_accent(best_text), 'embedding': json.dumps(emb),
        'total_searches': s['total'], 'searches_1d': s['c1d'],
        'searches_7d': s['c7d'], 'searches_15d': s['c15d'],
        'trending_multiplier': round(multiplier, 4),
        'is_trending': multiplier >= CONFIG['TRENDING_THRESHOLD'],
        'es_weight': round(final_score, 4),
        'last_searched_at': s['last'].strftime('%Y-%m-%d') if s['last'] else '',
    })

if suggestion_docs:
    max_score = max(d['es_weight'] for d in suggestion_docs)
    for d in suggestion_docs:
        nw = d['es_weight'] / max_score if max_score > 0 else 0
        d['es_weight'] = max(CONFIG['ES_WEIGHT_MIN'], min(CONFIG['ES_WEIGHT_MAX'],
            int(CONFIG['ES_WEIGHT_MIN'] + nw * (CONFIG['ES_WEIGHT_MAX'] - CONFIG['ES_WEIGHT_MIN']))))

print(f"\n{len(suggestion_docs)} suggestions")
for d in sorted(suggestion_docs, key=lambda x: x['es_weight'], reverse=True)[:10]:
    badge = "TREND" if d['is_trending'] else "     "
    print(f"[{badge}] [{d['es_weight']:5d}] {d['query_text']:28} | total={d['total_searches']:3d}")

Generating suggestion embeddings...


100%|██████████| 110/110 [00:01<00:00, 89.79it/s]


110 suggestions
[TREND] [10000] pho                          | total= 24
[TREND] [ 4635] poh                          | total= 12
[TREND] [ 3518] phở bò                       | total= 14
[TREND] [ 3069] bánh xèo                     | total= 25
[TREND] [ 2766] bánh mì                      | total= 45
[TREND] [ 1952] bún bò                       | total=  8
[TREND] [ 1911] gỏi bò                       | total= 30
[TREND] [ 1587] kem flan                     | total= 33
[TREND] [ 1545] tôm                          | total=  4
[TREND] [ 1545] p                            | total=  4


In [13]:
import csv
from google.colab import files

# Product embeddings
with open("embeddings.csv", "w", newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['product_id', 'embedding'])
    writer.writeheader()
    for _, row in df_products.iterrows():
        writer.writerow({'product_id': int(row['id']), 'embedding': json.dumps(row['embedding'])})
files.download("embeddings.csv")
print(f"Exported {len(df_products)} product embeddings")

# Suggestions
fieldnames = ['query_text','query_normalized','no_accent','embedding',
              'total_searches','searches_1d','searches_7d','searches_15d',
              'trending_multiplier','is_trending','es_weight','last_searched_at']
with open("suggestions.csv", "w", newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for d in suggestion_docs: writer.writerow(d)
files.download("suggestions.csv")
print(f"Exported {len(suggestion_docs)} suggestions")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported 474 product embeddings


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported 110 suggestions


In [22]:
#CELL14: Start Flask server + Cloudflare tunnel

from flask import Flask, request, jsonify
import threading

embed_app = Flask(__name__)
model.eval()

@embed_app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": CONFIG["MODEL_NAME"], "dim": CONFIG["VECTOR_DIM"]})

@embed_app.route("/embed", methods=["POST"])
def embed_endpoint():
    data = request.get_json()
    text = data.get("text", "").strip()
    if not text:
        return jsonify({"embedding": [0.0] * CONFIG["VECTOR_DIM"]})
    enc = tokenizer(text, max_length=CONFIG["MAX_LENGTH"], padding='max_length',
                    truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**enc)
    emb = nn.functional.normalize(mean_pooling(out, enc['attention_mask']), p=2, dim=1)
    return jsonify({"embedding": emb.cpu().numpy()[0].tolist()})

t = threading.Thread(target=lambda: embed_app.run(host='0.0.0.0', port=5001, threaded=False, use_reloader=False))
t.daemon = True; t.start()

import time; time.sleep(2)
print("Flask server running on port 5001")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5001 is in use by another program. Either identify and stop that program, or start the server with a different port.


Flask server running on port 5001


In [23]:
import subprocess, time, re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:5001'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

tunnel_url = None
for _ in range(60):
    line = proc.stdout.readline().decode('utf-8', errors='ignore')
    if line.strip(): print(line.strip())
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0); break
    time.sleep(1)

if tunnel_url:
    print(f"\n{'='*60}\nTUNNEL URL: {tunnel_url}\n{'='*60}")
    print(f"\nDán vào application.properties:")
    print(f"   phobert.server-url={tunnel_url}")
    print(f"   phobert.enabled=true")
else:
    print("Không lấy được URL, chạy lại cell này")

cloudflared: Text file busy
2026-04-20T08:24:31Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-20T08:24:31Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-20T08:24:34Z INF +--------------------------------------------------------------------------------------------+
2026-04-20T08:24:34Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-20T08:24:34Z INF |  https://twenty-mothers-su

In [24]:
import requests, numpy as np

resp = requests.get(f"{tunnel_url}/health", timeout=15)
print(f"Health: {resp.json()}")

emb1 = requests.post(f"{tunnel_url}/embed", json={"text": "pho bo"}, timeout=30).json()['embedding']
emb2 = requests.post(f"{tunnel_url}/embed", json={"text": "poh"}, timeout=30).json()['embedding']

cos_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
print(f"dim={len(emb1)}, cosine('pho bo', 'poh')={cos_sim:.4f}")
# Mong đợi > 0.7 — PhoBERT hiểu "poh" ≈ "phở bò"

INFO:werkzeug:127.0.0.1 - - [20/Apr/2026 08:24:38] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Apr/2026 08:24:39] "POST /embed HTTP/1.1" 200 -


Health: {'dim': 768, 'model': 'vinai/phobert-base-v2', 'status': 'ok'}


INFO:werkzeug:127.0.0.1 - - [20/Apr/2026 08:24:39] "POST /embed HTTP/1.1" 200 -


dim=768, cosine('pho bo', 'poh')=0.6582
